# Synthetic Data Generation v5 - Defects-Aware
## Replication of 4 Auditoria3 Sources with Ground Truth Defect Registry

Generates 4 independent tables with injected defects that match what the normalization pipeline validates and corrects:
- **TEXT_NORMALIZATION**: Spaces, punctuation, case inconsistencies (what `norm_text()` handles)
- **NUMERIC_FORMAT**: Non-numeric chars in DNI/matricula (what `norm_dni()`, `norm_matricula()` fix)
- **DATE_FORMAT**: Invalid formats, inconsistent separators (what `to_datetime(dayfirst=True)` parses)
- **MATCHING_DEFECTS**: Domain duplicates, liters discrepancies (what matching logic detects)
- **CONSISTENCY_ERRORS**: Cross-table logical inconsistencies (what pipeline validations catch)

Tracks every injected defect in `ground_truth.csv` for validation analysis.

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime

drive.mount('/content/drive')
print("✓ Google Drive mounted")

## Copy Generator Code
Paste the complete `generator_v5_defects_aware.py` code here:

In [ ]:
# INLINE GENERATOR CODE - paste full generator_v5_defects_aware.py here
# For now, we'll import it from the uploaded file

import sys
sys.path.insert(0, '/content/drive/MyDrive/Integrador')

# If generator uploaded separately, import it:
# from generator_v5_defects_aware import generate_dataset, GenerationConfig

# For Colab execution, paste the full generator_v5_defects_aware.py content here
# (Due to length, shown as placeholder)
exec(open('/content/drive/MyDrive/Integrador/generator_v5_defects_aware.py').read())

## Generate Dataset with Defects

In [ ]:
# Configure generation
config = GenerationConfig(
    num_vehicles=200,
    num_fuel_reports=400,
    num_fuel_requests=420,
    seed=20260816,
    scenario="defects_aware_v5",
    # Defect injection rates
    text_normalization_rate=0.08,
    numeric_format_rate=0.05,
    date_format_rate=0.04,
    matching_defect_rate=0.06,
    consistency_error_rate=0.03,
)

# Generate
tables = generate_dataset(config)

## Export to Google Drive

In [ ]:
# Export directory
export_dir = Path('/content/drive/MyDrive/Integrador/datasets/defects_aware_v5')
export_dir.mkdir(parents=True, exist_ok=True)

print(f"📁 Exporting to {export_dir}\n")

# Export each table
for table_name, df in tables.items():
    if table_name != 'ejecucion_dataset':
        output_file = export_dir / f"{table_name}.csv"
        df.to_csv(output_file, index=False)
        print(f"✓ {table_name:30s} → {output_file.name}")

# Export execution log
if 'ejecucion_dataset' in tables:
    ejecucion_file = export_dir / 'ejecucion_dataset.csv'
    tables['ejecucion_dataset'].to_csv(ejecucion_file, index=False)
    print(f"✓ {'ejecucion_dataset':30s} → {ejecucion_file.name}")

# Create manifest
manifest = {
    'generated_at': datetime.now().isoformat(),
    'seed': config.seed,
    'scenario': config.scenario,
    'num_vehicles': len(tables['vehiculo']),
    'num_devices': len(tables['dispositivo']),
    'num_fuel_reports': len(tables['reporte_consumo']),
    'num_fuel_requests': len(tables['solicitud_combustible']),
    'num_defects': len(tables['ground_truth']),
    'defect_rates': {
        'text_normalization': config.text_normalization_rate,
        'numeric_format': config.numeric_format_rate,
        'date_format': config.date_format_rate,
        'matching_defect': config.matching_defect_rate,
        'consistency_error': config.consistency_error_rate,
    },
    'config': {
        'num_vehicles': config.num_vehicles,
        'num_fuel_reports': config.num_fuel_reports,
        'num_fuel_requests': config.num_fuel_requests,
    }
}

manifest_file = export_dir / 'manifest.json'
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f"✓ {'manifest':30s} → manifest.json")

print(f"\n✅ All files exported to {export_dir}")

## Ground Truth Analysis

In [ ]:
# Analyze ground truth
gt = tables['ground_truth']

print(f"\n📊 GROUND TRUTH DEFECT REGISTRY")
print(f"\nTotal defects: {len(gt)}")
print(f"\nBY DEFECT TYPE:")
for dtype in gt['defect_type'].unique():
    count = len(gt[gt['defect_type'] == dtype])
    pct = 100 * count / len(gt)
    print(f"  {dtype:30s} {count:4d} ({pct:5.1f}%)")

print(f"\nBY SEVERITY:")
for sev in ['alta', 'media', 'baja']:
    count = len(gt[gt['severity'] == sev])
    if count > 0:
        pct = 100 * count / len(gt)
        print(f"  {sev:10s} {count:4d} ({pct:5.1f}%)")

print(f"\nBY TABLE:")
for table in sorted(gt['table'].unique()):
    count = len(gt[gt['table'] == table])
    pct = 100 * count / len(gt)
    print(f"  {table:30s} {count:4d} ({pct:5.1f}%)")

print(f"\nBY COLUMN (Top 10):")
for col, count in gt['column'].value_counts().head(10).items():
    print(f"  {col:30s} {count:4d}")

## Defect Examples

In [ ]:
print("\n📋 DEFECT EXAMPLES (First 20):")
print("\n")

for idx, row in gt.head(20).iterrows():
    print(f"{row['row_id']:30s} | {row['table']:20s} | {row['column']:30s}")
    print(f"  Type: {row['defect_type']:30s} Severity: {row['severity']:6s}")
    print(f"  Description: {row['description']}")
    if pd.notna(row['original_value']):
        print(f"  Original → Injected: '{row['original_value']}' → '{row['injected_value']}'")
    print()

## Dataset Quality Summary

In [ ]:
print("\n" + "="*100)
print("📊 DATASET QUALITY SUMMARY")
print("="*100)

print(f"\n📈 VOLUME:")
for name, df in tables.items():
    if name != 'ejecucion_dataset':
        print(f"  {name:30s} {len(df):8d} records × {len(df.columns):2d} fields")

total_records = sum(len(df) for name, df in tables.items() if name != 'ejecucion_dataset')
print(f"  {'TOTAL':30s} {total_records:8d} records")

print(f"\n🔴 DEFECTS INJECTED:")
print(f"  Total defects: {len(gt)}")
print(f"  Coverage: {100*len(gt)/total_records:.2f}% of records have defects")

print(f"\n🎯 PIPELINE ALIGNMENT:")
print(f"  Text normalization (norm_text): {len(gt[gt['defect_type']=='TEXT_NORMALIZATION'])} defects")
print(f"  Numeric format (norm_dni/matricula): {len(gt[gt['defect_type']=='NUMERIC_FORMAT'])} defects")
print(f"  Date parsing (to_datetime): {len(gt[gt['defect_type']=='DATE_FORMAT'])} defects")
print(f"  Matching logic: {len(gt[gt['defect_type']=='MATCHING_DEFECT'])} defects")
print(f"  Consistency checks: {len(gt[gt['defect_type']=='CONSISTENCY_ERROR'])} defects")

print("\n" + "="*100)